# VAE: Auto-Encoding Variational Bayes (Kingma & Welling, ICLR 2014)
### Triển Khai Từ Con Số 0 (From Scratch) Bằng Python & NumPy

> **Tác giả:** Diederik P. Kingma, Max Welling  
> **Hội nghị:** ICLR 2014 (Banff, Canada)  
> **Ý nghĩa lịch sử:** Kết hợp xuất sắc giữa Học Sâu và Thống Kê Bayes Biến Phân. Đưa ra Thủ thuật Tái Tham Số Hóa (Reparameterization Trick), biến Autoencoder thành Mô Hình Tạo Sinh (Generative Model) hoàn chỉnh.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Reparameterization Trick:** Lập trình $z = \mu + \sigma \odot \epsilon$ với $\epsilon \sim \mathcal{N}(0, I)$ và lan truyền ngược giải tích qua biến ngẫu nhiên.
2. **Hàm mất mát ELBO:** Cài đặt hai thành phần cốt lõi: Reconstruction Loss (BCE) và KL Divergence giải tích.
3. **Tạo sinh ngẫu nhiên:** Lấy mẫu $z \sim \mathcal{N}(0, I)$ và giải mã để sinh mẫu dữ liệu mới chưa từng có.
4. **Academic Value Benchmark:** Đối chiếu định lượng sự khác biệt bản chất giữa Autoencoder Cổ Điển (Deterministic AE) và VAE.

In [ ]:
import time
import numpy as np
print(f"[✓] Khởi tạo thư viện NumPy: {np.__version__}")

## PHẦN 1: CÁC KHỐI TẦNG NƠ-RON CƠ BẢN (FORWARD & BACKWARD)

Cài đặt các tầng tuyến tính `LinearLayer`, hàm kích hoạt `ReLULayer` và `SigmoidLayer` có đạo hàm giải tích chuẩn xác.

In [ ]:
class LinearLayer:
    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features).astype(np.float32) * scale
        self.b = np.zeros((1, out_features), dtype=np.float32)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.x = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, grad_output):
        self.dW = np.dot(self.x.T, grad_output)
        self.db = np.sum(grad_output, axis=0, keepdims=True)
        return np.dot(grad_output, self.W.T)


class ReLULayer:
    def __init__(self):
        self.x = None

    def forward(self, x):
        self.x = x
        return np.maximum(0.0, x)

    def backward(self, grad_output):
        return grad_output * (self.x > 0).astype(np.float32)


class SigmoidLayer:
    def __init__(self):
        self.out = None

    def forward(self, x):
        x_clipped = np.clip(x, -15.0, 15.0)
        self.out = 1.0 / (1.0 + np.exp(-x_clipped))
        return self.out

    def backward(self, grad_output):
        return grad_output * self.out * (1.0 - self.out)

print("[✓] Đã khởi tạo LinearLayer, ReLULayer, và SigmoidLayer.")

## PHẦN 2: THỦ THUẬT TÁI THAM SỐ HÓA & HÀM MỤC TIÊU ELBO

1. **Reparameterization Trick:** Tách tính ngẫu nhiên sang biến nhiễu ngoại sinh $\epsilon \sim \mathcal{N}(0, I)$:
$$z = \mu + \exp(0.5 \cdot \log\sigma^2) \odot \epsilon$$

2. **Độ phân kỳ KL Divergence giải tích:**
$$D_{\text{KL}}(q_\phi(z|x) \parallel \mathcal{N}(0, I)) = -\frac{1}{2} \sum_{j=1}^J \left( 1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2 \right)$$

In [ ]:
class ReparameterizationTrick:
    def __init__(self):
        self.mu = None
        self.log_var = None
        self.eps = None
        self.std = None

    def forward(self, mu, log_var):
        self.mu = mu
        self.log_var = log_var
        self.std = np.exp(0.5 * log_var)
        self.eps = np.random.randn(*mu.shape).astype(np.float32)
        return self.mu + self.std * self.eps

    def backward(self, grad_z):
        grad_mu = grad_z
        grad_log_var = grad_z * (self.eps * 0.5 * self.std)
        return grad_mu, grad_log_var


def compute_elbo_loss_and_gradients(x_true, x_pred, mu, log_var):
    N = x_true.shape[0]
    eps = 1e-12

    # 1. Reconstruction Loss (BCE)
    x_pred_safe = np.clip(x_pred, eps, 1.0 - eps)
    bce = -np.sum(x_true * np.log(x_pred_safe) + (1.0 - x_true) * np.log(1.0 - x_pred_safe)) / N
    grad_logits = (x_pred - x_true) / N

    # 2. KL Divergence Loss
    kl_element = 1.0 + log_var - mu**2 - np.exp(log_var)
    kl_loss = -0.5 * np.sum(kl_element) / N
    grad_mu_kl = mu / N
    grad_log_var_kl = -0.5 * (1.0 - np.exp(log_var)) / N

    total_loss = bce + kl_loss
    return total_loss, bce, kl_loss, grad_logits, grad_mu_kl, grad_log_var_kl

print("[✓] Đã khởi tạo ReparameterizationTrick và hàm tính mất mát ELBO.")

## PHẦN 3: KIẾN TRÚC MÔ HÌNH VAE TỪ CON SỐ 0

Lắp ráp trọn vẹn kiến trúc VAE gồm Encoder ($x \to \mu, \log\sigma^2$), bộ tái tham số hóa và Decoder ($z \to \hat{x}$).

In [ ]:
from vae_scratch import VAEScratch, DeterministicAutoencoderScratch

vae = VAEScratch(input_dim=64, hidden_dim=32, latent_dim=4, lr=0.05)
ae = DeterministicAutoencoderScratch(input_dim=64, hidden_dim=32, latent_dim=4, lr=0.08)
print("[✓] Đã khởi tạo VAEScratch và DeterministicAutoencoderScratch.")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

So sánh khả năng sinh dữ liệu mới từ không gian ẩn:
- **Autoencoder Cũ:** Lấy mẫu $z \sim \mathcal{N}(0, I)$ sinh ra mẫu nhiễu vô nghĩa do các vùng rỗng không được điều hòa.
- **Variational Autoencoder (VAE):** Không gian ẩn phủ kín phân phối Gauss, sinh mẫu chuẩn xác và sắc nét.

In [ ]:
from vae_scratch import run_academic_value_comparison

# Chạy kiểm thử đối chuẩn học thuật trực tiếp
run_academic_value_comparison()

## 🎓 TỔNG KẾT & BÀI HỌC LỊCH SỬ TỪ VAE (ICLR 2014)

1. **Khai sinh Kỷ nguyên AI Tạo Sinh Hiện Đại:** VAE chứng minh mạng nơ-ron sâu có thể học các phân phối xác suất phức tạp và sinh ra dữ liệu mới hoàn toàn.
2. **Reparameterization Trick:** Một trong những thủ thuật toán học kinh điển và thanh lịch nhất của Học Sâu, cho phép gradient lan truyền ngược qua các phân phối ngẫu nhiên.
3. **Cân bằng ELBO:** Cơ chế giằng co hoàn hảo giữa độ sắc nét của phép tái tạo và độ trơn tru, liền mạch của không gian ẩn Gauss.
4. **Di sản vượt thời gian:** Nền tảng cốt lõi của Conditional VAE, $\beta$-VAE, VQ-VAE, và đặc biệt là không gian ẩn của các mô hình khuếch tán (Latent Diffusion Models / Stable Diffusion).